# Handoffs

*Notebook 14*

Route tasks to the right specialist automatically using the handoff pattern.
<br>
<br>
**Topics:**

- Why one agent isn't always enough

- The triage → specialist pattern

- Configuring handoffs with `handoffs=[]`

- Customer support example: triage → billing → tech support → refunds

- Using the `handoff()` function with custom `tool_description_override`

- Tracking which agent handled the response

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, handoff

MODEL = "gpt-5-mini"


print("✅ Ready!")

---

## 🎯 The Problem

A single agent handling both billing and technical support makes compromises everywhere.

Specialists with focused instructions, knowledge, and tone consistently outperform a generalist trying to do both.

---

## 🔄 Part 1: How Handoffs Work

A handoff is a transfer of responsibility from one agent to another.

The SDK turns each handoff into a tool the triage agent can call — for example, `transfer_to_billing_agent`.

When the triage agent calls that tool, the specialist takes over the conversation with full context.

The triage agent does not get control back.

This is intentional — the specialist owns the conversation once it starts, so triage can't re-route or second-guess.

If the specialist needs more information, it asks the user directly.

```
User message
     ↓
Triage Agent   ← decides who should handle this
     ↓ handoff
Specialist Agent  ← handles the request and responds
```

---

## 🏥 Part 2: A Simple Triage System

We'll build a customer support system with a triage agent and three specialists (billing, technical support, and refunds).

The agent `name` becomes part of the auto-generated handoff tool name (e.g., `BillingAgent` → `transfer_to_billing_agent`), so pick names that read well as routing destinations.

In [ ]:
billing_instructions = (
    "You are a billing specialist. You help with:\n"
    "- Invoice questions and payment history\n"
    "- Subscription changes and cancellations\n"
    "- Refund requests and billing errors\n"
    "Be empathetic and solution-focused. Always offer a clear next step."
)

billing_agent = Agent(
    name="BillingAgent",
    instructions=billing_instructions,
    model=MODEL
)

tech_instructions = (
    "You are a technical support specialist. You help with:\n"
    "- Setup and installation issues\n"
    "- Bug reports and error messages\n"
    "- Feature questions and how-to guidance\n"
    "Be precise and step-by-step in your responses."
)

tech_agent = Agent(
    name="TechSupportAgent",
    instructions=tech_instructions,
    model=MODEL
)

refunds_instructions = (
    "You are a refunds specialist. You help with:\n"
    "- Processing refund requests\n"
    "- Refund status and timelines\n"
    "- Eligibility questions and exceptions\n"
    "Be clear about timelines and set accurate expectations."
)

refunds_agent = Agent(
    name="RefundsAgent",
    instructions=refunds_instructions,
    model=MODEL
)

# --------------------------------------------------------------
print("✅ Specialist agents defined")

#### Define the Triage Agent

In [ ]:
triage_instructions = (
    "You are a customer support triage agent. Your only job is to route requests.\n\n"
    "- Billing questions (invoices, payments, subscriptions) → hand off to BillingAgent\n"
    "- Technical questions (bugs, setup, errors, features) → hand off to TechSupportAgent\n"
    "- Refund requests and refund status → hand off to RefundsAgent\n\n"
    "Do not answer questions yourself. Always hand off to the right specialist."
)

triage_agent = Agent(
    name="TriageAgent",
    instructions=triage_instructions,
    model=MODEL,
    handoffs=[billing_agent, tech_agent, refunds_agent]
)

# --------------------------------------------------------------
print("✅ Triage agent ready")
print("   Handoffs: BillingAgent, TechSupportAgent, RefundsAgent")

### 💡 Why This Works

The SDK automatically creates `transfer_to_billing_agent`, `transfer_to_tech_support_agent`, and `transfer_to_refunds_agent` tools for the triage agent.

It calls whichever one matches the request — no routing logic to write yourself.

Routing is prompt-driven, not enforced — clear, narrow triage instructions are what keep the agent from answering directly.

If `result.last_agent.name` is still the triage agent, the handoff didn't happen.

---

## 🧪 Part 3: See the Handoffs in Action

The `RunResult` returned by `Runner.run` has a `last_agent` attribute — `result.last_agent.name` tells you which agent produced the final response.

This is how you debug routing mistakes and verify the triage agent is sending requests to the right specialist.

In [ ]:
test_messages = [
    "I was charged twice for my subscription this month.",
    "My app crashes when I try to export a PDF.",
    "I can't get the desktop app to install on Windows.",
    "I'd like to request a refund for last month's charge."
]

# --------------------------------------------------------------
print("✅ Test messages ready")

#### Run All Test Messages

In [ ]:
print("🔄 HANDOFF DEMO")
print("=" * 60)

for message in test_messages:

    result = await Runner.run(triage_agent, input=message)

    print(f"\n📨 Message: {message}")
    print(f"🤖 Handled by: {result.last_agent.name}")
    print(f"Response: {result.final_output}")
    print("-" * 60)

---

## ⚙️ Part 4: Customizing Handoffs

Use the `handoff()` function when you need more control over how each handoff is described to the triage agent — a custom description tells it exactly when to use each specialist.

In [ ]:
triage_v2_instructions = (
    "You are a customer support triage agent. Route every request to the right specialist.\n"
    "Do not answer questions yourself."
)

triage_agent_v2 = Agent(
    name="TriageAgentV2",
    instructions=triage_v2_instructions,
    model=MODEL,
    handoffs=[
        handoff(
            billing_agent,
            tool_description_override="Transfer to billing for invoice, payment, or subscription questions."
        ),
        handoff(
            tech_agent,
            tool_description_override="Transfer to tech support for bugs, errors, installation, or feature questions."
        ),
        handoff(
            refunds_agent,
            tool_description_override="Transfer to refunds for refund requests, refund status, and eligibility questions."
        )
    ]
)

# --------------------------------------------------------------
print("✅ TriageAgentV2 ready with custom handoff descriptions")

#### Compare Routing on an Ambiguous Message

You may need to run this a few times — on borderline messages like this one, v1 routing can vary while v2 stays consistent.

In [ ]:
ambiguous_message = "I need help with my account settings."

print("🔄 BEFORE/AFTER COMPARISON")
print("=" * 60)

result_v1 = await Runner.run(triage_agent, input=ambiguous_message)

print(f"📨 Message: {ambiguous_message}")
print(f"🤖 v1 (simple handoffs) routed to: {result_v1.last_agent.name}")

result_v2 = await Runner.run(triage_agent_v2, input=ambiguous_message)

print(f"🤖 v2 (custom descriptions) routed to: {result_v2.last_agent.name}")
print("=" * 60)

### 💡 Why This Works

Custom `tool_description_override` text gives the triage agent more explicit routing criteria — on clearly-scoped requests both versions usually agree, but on ambiguous requests the override is what makes routing predictable.

Use plain `handoffs=[...]` when the agent names already make the routing obvious — reach for `handoff(..., tool_description_override=...)` when requests could plausibly fit two specialists.

---

## 💪 Practice Exercises

### Exercise 1: Four-Way Routing

*Covers: handoffs — extending a triage routing system*

Add a fourth specialist — an FAQ agent — and update the triage agent to route to all four.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Four-Way Routing
# --------------------------------------------------------------
# Objective: Add an FAQ agent and update triage to route to all four.

# TODO 1: Create an FAQ agent with instructions to answer general
#          questions about pricing, features, and company policies

# TODO 2: Create a new triage agent with handoffs to:
#          billing_agent, tech_agent, refunds_agent, and your new faq_agent
#          Use handoff() with tool_description_override for all four

# TODO 3: Test with these messages and print result.last_agent.name:
#          - "What is your pricing?"
#          - "I can't log in"
#          - "When will my refund arrive?"

# --- Write your code below this line ---

### Exercise 2: Domain-Specific Specialist

*Covers: handoffs — building a domain-specific routing system*

Build a different triage system for a different domain — a recipe assistant that routes to a vegetarian specialist or a meat specialist.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Recipe Assistant
# --------------------------------------------------------------
# Objective: Build a triage system for a different domain.

# TODO 1: Create a VegetarianChef agent with relevant instructions

# TODO 2: Create a MeatChef agent with relevant instructions

# TODO 3: Create a RecipeTriage agent that routes to the right chef

# TODO 4: Test with: "How do I make a great burger?"
#         and: "What's a good lentil dish?"
#         Print result.last_agent.name and result.final_output

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Handoffs transfer responsibility:**

- The specialist takes over completely — triage does not get control back

- The specialist receives the full conversation history for context
<br>
<br>

**Two ways to configure:**

- `handoffs=[agent]` — simple; the SDK auto-generates a tool name based on the agent name

- `handoffs=[handoff(agent, tool_description_override="...")]` — explicit routing hints for better decisions
<br>
<br>

**Track what happened:**

- `result.last_agent.name` shows which agent produced the final response

- Check the tracing dashboard (covered in Lesson 25) to see the full handoff chain

---

## 📍 Next Step

**Notebook 15: Agents as Tools**  

Learn when to call an agent like a function instead of handing off to it — and why the distinction matters.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-14-handoffs)

---